# ML 01 – Uber Fare Prediction

Run the cells from top to bottom. Update dataset paths or column names if your CSV uses different names.

## Dataset
Place `uber.csv` in `../datasets/` (or change `DATA_PATH`). Expected columns:
`fare_amount`, `pickup_longitude`, `pickup_latitude`, `dropoff_longitude`, `dropoff_latitude`, `passenger_count`.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

DATA_PATH = "../datasets/uber.csv"
df = pd.read_csv(DATA_PATH)
print(df.head())
print(df.info())

In [ ]:
required = ["fare_amount", "pickup_longitude", "pickup_latitude",
            "dropoff_longitude", "dropoff_latitude", "passenger_count"]
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Missing columns: {missing}. Update the column mapping for your dataset.")

df = df[required].dropna()
# Basic plausibility filtering; adjust thresholds to match the lab dataset.
df = df[(df.fare_amount > 0) & (df.fare_amount < 500)]
df = df[df.passenger_count.between(1, 6)]
df = df[df.pickup_longitude.between(-180, 180) & df.dropoff_longitude.between(-180, 180)]
df = df[df.pickup_latitude.between(-90, 90) & df.dropoff_latitude.between(-90, 90)]

X = df.drop(columns="fare_amount")
y = df["fare_amount"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print("Training rows:", len(X_train), "Testing rows:", len(X_test))

In [ ]:
models = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
}
for name, model in models.items():
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, predictions))
    print(f"{name}: R2={r2_score(y_test, predictions):.4f}, RMSE={rmse:.4f}")

## Notes
This is a baseline using the listed numeric columns. For stronger fare prediction, derive trip distance and clean geographic outliers. Results depend on the dataset and filtering thresholds.